# 05 — Multimodal Alignment: BLIP-2 and Q-Former

## What
BLIP-2 (Bootstrapping Language-Image Pretraining 2, Li et al. 2023) introduced the Q-Former — a lightweight transformer with a fixed set of learnable query vectors that attend to frozen image features to extract the most LLM-relevant visual information. The Q-Former is the only trained component; both the ViT and LLM are frozen.

## Why
Training a full VLM end-to-end requires massive compute. The Q-Former bottleneck compresses 256 patch embeddings into 32 fixed-length query tokens, dramatically reducing the compute needed at the LLM interface. It also separates visual representation learning from language grounding into two pre-training stages.

## Community context
BLIP-2 achieves competitive VQA performance with 5x less training time than Flamingo. InstructBLIP added instruction tuning on top of the Q-Former, dramatically improving zero-shot generalisation. MiniGPT-4 showed you can fine-tune only a linear projection layer on top of BLIP-2's Q-Former with ~5 hours of GPU time.

In [ ]:
# Q-Former: learned queries attend to image patches
import numpy as np

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

class QFormer:
    """
    Simplified Q-Former: N_q learnable query vectors attend to image patches.
    Output: N_q vectors that summarise the image for the LLM.
    Two attention operations:
      1. Query-image cross-attention: queries extract from visual patches
      2. Query self-attention: queries can share info with each other
    """
    def __init__(self, n_queries=32, vis_dim=256, query_dim=256, n_heads=4):
        self.n_queries = n_queries
        self.query_dim = query_dim
        scale = np.sqrt(query_dim)
        # Learnable query vectors (initialised randomly, trained end-to-end)
        self.queries = np.random.randn(n_queries, query_dim) * 0.02
        # Cross-attention: queries -> image
        self.Wq_cross = np.random.randn(query_dim, query_dim) / scale
        self.Wk_cross = np.random.randn(vis_dim, query_dim)   / scale
        self.Wv_cross = np.random.randn(vis_dim, query_dim)   / scale
        # Self-attention among queries
        self.Wq_self  = np.random.randn(query_dim, query_dim) / scale
        self.Wk_self  = np.random.randn(query_dim, query_dim) / scale
        self.Wv_self  = np.random.randn(query_dim, query_dim) / scale
        # Output projection to LLM dimension
        self.head_dim = query_dim // n_heads

    def cross_attend(self, queries, image_features):
        Q = queries       @ self.Wq_cross   # (N_q, D)
        K = image_features @ self.Wk_cross  # (N_p, D)
        V = image_features @ self.Wv_cross
        attn = softmax(Q @ K.T / np.sqrt(self.head_dim))
        return attn @ V   # (N_q, D)

    def self_attend(self, queries):
        Q = queries @ self.Wq_self
        K = queries @ self.Wk_self
        V = queries @ self.Wv_self
        attn = softmax(Q @ K.T / np.sqrt(self.head_dim))
        return attn @ V

    def __call__(self, image_features):
        """image_features: (N_patches, vis_dim) -> output: (N_q, query_dim)"""
        # Stage 1: extract visual info into query slots
        q = self.queries + self.cross_attend(self.queries, image_features)
        # Stage 2: let queries communicate
        q = q + self.self_attend(q)
        return q

np.random.seed(42)
qformer = QFormer(n_queries=32, vis_dim=256, query_dim=256)

# Simulate 256 image patches from a frozen ViT
image_patches = np.random.randn(256, 256)
query_output  = qformer(image_patches)

print('Q-Former compression:')
print(f'  Input:  {image_patches.shape}  (256 ViT patch tokens)')
print(f'  Output: {query_output.shape}   (32 compressed query tokens)')
print(f'  Compression ratio: {256//32}x')
print(f'\nQuery token stats: mean={query_output.mean():.4f} std={query_output.std():.4f}')
print('These 32 tokens feed into the LLM, not the 256 raw patches')
print('-> LLM computational cost reduced 8x at vision-language boundary')

## BLIP-2 Two-Stage Pretraining

Stage 1 trains the Q-Former to learn image-text representations (ITC, ITM, ITG losses). Stage 2 trains the Q-Former to feed into a frozen LLM for generation. Only the Q-Former weights change — both ViT and LLM are fully frozen throughout.

In [ ]:
# Simulate BLIP-2 pretraining losses
def image_text_contrastive_loss(q_features, text_features, temperature=0.07):
    """ITC: align Q-Former outputs with text representations"""
    q_pool = q_features.mean(axis=0)  # pool over queries
    q_pool = q_pool / (np.linalg.norm(q_pool) + 1e-8)
    txt = text_features / (np.linalg.norm(text_features) + 1e-8)
    similarity = np.dot(q_pool, txt) / temperature
    return -similarity  # maximise similarity -> minimise negative

def image_text_matching_loss(q_features, text_features, matched):
    """ITM: binary classification of matched vs mismatched pairs"""
    # Simulate a small MLP on top of query+text features
    combined = np.concatenate([q_features.mean(0), text_features])
    # Fake logit (positive if matched, negative if not)
    logit = np.dot(q_features.mean(0), text_features) * (1 if matched else -1)
    prob = 1 / (1 + np.exp(-logit))
    target = 1.0 if matched else 0.0
    return -(target * np.log(prob+1e-10) + (1-target)*np.log(1-prob+1e-10))

np.random.seed(5)
q_out = qformer(image_patches)
text_feat = np.random.randn(256)  # from BERT text encoder
text_unrelated = np.random.randn(256)

itc = image_text_contrastive_loss(q_out, text_feat)
itm_pos = image_text_matching_loss(q_out, text_feat, matched=True)
itm_neg = image_text_matching_loss(q_out, text_unrelated, matched=False)

print('BLIP-2 Stage 1 losses (representation learning):')
print(f'  ITC loss (image-text alignment): {itc:.4f}')
print(f'  ITM loss (matched pair):          {itm_pos:.4f}')
print(f'  ITM loss (mismatched pair):       {itm_neg:.4f}')
print('\nStage 2: Q-Former outputs fed to frozen LLM for autoregressive generation')